# 📄 OmniFile AI Processor v4.0
## دفتر العمل السحابي الكامل — Complete Cloud Workbook

---

### 🔍 Multilingual OCR · Translation · Spell Correction

**Arabic | English | German | French | Urdu | +30 languages**

---

### 👨‍💻 **Author:** Dr. Abdulmalek

| Resource | Link |
|---|---|
| **GitHub Repository** | [https://github.com/DrAbdulmalek/OmniFile_Processor](https://github.com/DrAbdulmalek/OmniFile_Processor) |
| **HuggingFace Spaces** | [https://huggingface.co/spaces/DrAbdulmalek/handwriting-ocr](https://huggingface.co/spaces/DrAbdulmalek/handwriting-ocr) |

---

### ✨ Features

- **📄 PDF Processing** — Extract text from PDF files using PyMuPDF
- **🖼️ Image OCR** — EasyOCR + TrOCR for printed & handwritten text
- **✏️ Spell Correction** — Arabic & English correction with custom dictionaries
- **🌐 Translation** — English ↔ Arabic using Helsinki-NLP models
- **📊 Evaluation** — CER/WER metrics with visualization
- **🖥️ Gradio UI** — Full web interface with 7 tabs + ngrok tunnel

---

> **📌 Note:** This notebook is optimized for Google Colab with GPU runtime.
> Go to **Runtime → Change runtime type → T4 GPU** for best performance.


In [ ]:
# %% [markdown]
# ## 1️⃣ Environment Setup
# Check GPU, install system packages, and prepare the Python environment.

# %%
import subprocess
import sys
import os
from pathlib import Path
import json
import time

print("=" * 70)
print("  🔧 OmniFile AI Processor v4.0 — Environment Setup")
print("=" * 70)

# ─── Check GPU Availability ───────────────────────────────────────────
print("\n🎮 GPU Availability:")
gpu_available = False
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
        print(f"   ✅ GPU: {gpu_name}")
        print(f"   ✅ VRAM: {gpu_mem:.2f} GB")
        print(f"   ✅ CUDA: {torch.version.cuda}")
        print(f"   ✅ PyTorch: {torch.__version__}")
        gpu_available = True
        device = "cuda"
    else:
        print("   ⚠️  No GPU detected — using CPU (slower)")
        device = "cpu"
except ImportError:
    print("   ⚠️  PyTorch not installed yet")
    device = "cpu"

# ─── Install System Packages ──────────────────────────────────────────
print("\n📦 Installing system packages...")
system_packages = [
    "poppler-utils",
    "tesseract-ocr",
    "tesseract-ocr-ara",
    "tesseract-ocr-eng",
    "tesseract-ocr-deu",
    "tesseract-ocr-fra",
    "tesseract-ocr-urd",
]

for pkg in system_packages:
    result = subprocess.run(
        ["apt-get", "install", "-y", "-q", pkg],
        capture_output=True, text=True
    )
    status = "✅" if result.returncode == 0 else "❌"
    print(f"   {status} {pkg}")

# ─── Verify Tesseract ────────────────────────────────────────────────
print("\n🔍 Verifying Tesseract OCR:")
try:
    result = subprocess.run(["tesseract", "--version"], capture_output=True, text=True)
    first_line = result.stdout.strip().split('\n')[0]
    print(f"   ✅ {first_line}")
    result = subprocess.run(["tesseract", "--list-langs"], capture_output=True, text=True)
    langs = result.stdout.strip().split('\n')[1:]
    print(f"   📋 Languages: {', '.join(langs[:10])}")
except FileNotFoundError:
    print("   ❌ Tesseract not found")

# ─── Install Python Packages ──────────────────────────────────────────
print("\n🐍 Installing Python packages...")

python_packages = [
    "easyocr",
    "transformers>=4.35.0",
    "pymupdf",
    "ar-corrector",
    "pyspellchecker",
    "sacrebleu",
    "jiwer",
    "gradio>=4.0.0",
    "pyngrok",
    "Pillow",
    "langdetect",
    "nltk",
    "matplotlib",
    "seaborn",
    "pandas",
    "numpy",
    "tqdm",
    "accelerate",
    "sentencepiece",
    "protobuf",
]

for pkg in python_packages:
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", pkg],
            capture_output=True, text=True, timeout=120
        )
        print(f"   ✅ {pkg}")
    except Exception as e:
        print(f"   ❌ {pkg}: {e}")

# ─── Display Environment Info ────────────────────────────────────────
print("\n" + "=" * 70)
print("  📊 Environment Summary")
print("=" * 70)
print(f"   Python:        {sys.version.split()[0]}")
print(f"   Device:        {device}")
print(f"   Working Dir:   {os.getcwd()}")
print(f"   Packages:      {len(python_packages)} installed")
print("=" * 70)
print("   ✅ Environment ready!\n")


In [ ]:
# %% [markdown]
# ## 2️⃣ Google Drive Integration
# Mount Drive, create project structure, and configure environment.

# %%
from google.colab import drive
import os
from pathlib import Path

print("=" * 70)
print("  📂 Google Drive Integration")
print("=" * 70)

# ─── Mount Google Drive ───────────────────────────────────────────────
print("\n🔌 Mounting Google Drive...")
drive.mount('/content/drive')

# ─── Define Project Paths ─────────────────────────────────────────────
PROJECT_ROOT = Path("/content/drive/MyDrive/OmniFile_AI")

DIRS = {
    "raw_pdfs":      PROJECT_ROOT / "data" / "raw" / "pdfs",
    "raw_images":    PROJECT_ROOT / "data" / "raw" / "images",
    "processed":     PROJECT_ROOT / "data" / "processed",
    "exports":       PROJECT_ROOT / "data" / "exports",
    "models_cache":  PROJECT_ROOT / "models_cache",
    "database":      PROJECT_ROOT / "database",
    "logs":          PROJECT_ROOT / "logs",
}

# ─── Create Directory Structure ───────────────────────────────────────
print("\n📁 Creating project directory structure...")
created_dirs = []
for name, path in DIRS.items():
    path.mkdir(parents=True, exist_ok=True)
    created_dirs.append(f"   ✅ {path}")
    print(f"   ✅ {path}")

# Create .gitkeep files so empty dirs persist in Drive
for name, path in DIRS.items():
    gitkeep = path / ".gitkeep"
    if not any(path.iterdir()):
        gitkeep.touch(exist_ok=True)

# ─── Set Environment Variables ────────────────────────────────────────
print("\n🔧 Configuring environment variables...")

# Set HuggingFace token (replace with your token or set in Colab secrets)
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    print("   ℹ️  No HF_TOKEN found. Set it via:")
    print("      os.environ['HF_TOKEN'] = 'your_token_here'")
    print("      Or use Colab Secrets: 🗝️ icon in the sidebar")
else:
    print(f"   ✅ HF_TOKEN configured (length={len(HF_TOKEN)})")

# Set Transformers cache to Drive (saves re-downloading models)
TRANSFORMERS_CACHE = str(DIRS["models_cache"])
os.environ["TRANSFORMERS_CACHE"] = TRANSFORMERS_CACHE
os.environ["HF_HOME"] = TRANSFORMERS_CACHE
print(f"   ✅ TRANSFORMERS_CACHE → {TRANSFORMERS_CACHE}")

# ─── Verify Drive Connection ──────────────────────────────────────────
print("\n🔍 Verifying Drive connection...")
if PROJECT_ROOT.exists():
    files = list(PROJECT_ROOT.rglob("*"))
    print(f"   ✅ Project root accessible")
    print(f"   📁 Total items in project: {len(files)}")

    # Check for existing PDFs
    pdfs = list(DIRS["raw_pdfs"].glob("*.pdf"))
    if pdfs:
        print(f"   📄 Found {len(pdfs)} PDF file(s):")
        for pdf in pdfs[:5]:
            size_mb = pdf.stat().st_size / 1e6
            print(f"      • {pdf.name} ({size_mb:.2f} MB)")

    # Check for existing images
    images = list(DIRS["raw_images"].glob("*.*"))
    if images:
        print(f"   🖼️  Found {len(images)} image file(s):")
        for img in images[:5]:
            size_kb = img.stat().st_size / 1e3
            print(f"      • {img.name} ({size_kb:.1f} KB)")
else:
    print("   ⚠️  Project root not found. Creating fresh structure.")

print("\n" + "=" * 70)
print("  ✅ Google Drive integration complete!")
print("=" * 70 + "\n")

# ─── Helper: Save results to Drive ────────────────────────────────────
def save_to_drive(data, filename, subdir="processed"):
    """Save data (text or JSON) to the Drive project folder."""
    save_path = DIRS[subdir] / filename
    if isinstance(data, (dict, list)):
        import json
        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
    else:
        with open(save_path, "w", encoding="utf-8") as f:
            f.write(str(data))
    print(f"   💾 Saved → {save_path}")
    return str(save_path)


In [ ]:
# %% [markdown]
# ## 3️⃣ Initialize OCR Engines
# Load EasyOCR and TrOCR models. Test with a sample image.

# %%
import torch
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import time
import gc

print("=" * 70)
print("  🔍 Initializing OCR Engines")
print("=" * 70)

# ─── Configuration ────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n⚙️  Using device: {device}")

OCR_ENGINES = {}

# ─── 1. EasyOCR ───────────────────────────────────────────────────────
print("\n📦 Loading EasyOCR (Arabic + English + German)...")
try:
    import easyocr
    start_time = time.time()
    easy_reader = easyocr.Reader(
        ['ar', 'en', 'de'],
        gpu=(device == "cuda"),
        download_enabled=True,
        model_storage_directory=str(DIRS["models_cache"]),
    )
    load_time = time.time() - start_time
    OCR_ENGINES["easyocr"] = {
        "model": easy_reader,
        "status": "✅ Ready",
        "languages": ["Arabic", "English", "German"],
        "type": "Multi-language (printed + handwritten)",
        "load_time": f"{load_time:.1f}s",
    }
    print(f"   ✅ EasyOCR loaded in {load_time:.1f}s")
    print(f"   📋 Languages: Arabic, English, German")
except Exception as e:
    OCR_ENGINES["easyocr"] = {"model": None, "status": f"❌ Error: {e}"}
    print(f"   ❌ EasyOCR failed: {e}")

# ─── 2. TrOCR (Transformer-based OCR) ────────────────────────────────
print("\n📦 Loading TrOCR (microsoft/trocr-base-handwritten)...")
try:
    from transformers import TrOCRProcessor, VisionEncoderDecoderModel
    start_time = time.time()

    trocr_processor = TrOCRProcessor.from_pretrained(
        "microsoft/trocr-base-handwritten",
        cache_dir=str(DIRS["models_cache"]),
    )
    trocr_model = VisionEncoderDecoderModel.from_pretrained(
        "microsoft/trocr-base-handwritten",
        cache_dir=str(DIRS["models_cache"]),
    ).to(device)
    load_time = time.time() - start_time

    OCR_ENGINES["trocr"] = {
        "model": trocr_model,
        "processor": trocr_processor,
        "status": "✅ Ready",
        "languages": ["English (handwriting-focused)"],
        "type": "Handwritten text recognition",
        "load_time": f"{load_time:.1f}s",
    }
    print(f"   ✅ TrOCR loaded in {load_time:.1f}s")
    print(f"   📋 Specialized for: Handwritten English text")
except Exception as e:
    OCR_ENGINES["trocr"] = {"model": None, "status": f"❌ Error: {e}"}
    print(f"   ❌ TrOCR failed: {e}")

# ─── 3. Tesseract (System) ────────────────────────────────────────────
print("\n📦 Checking Tesseract OCR (system)...")
try:
    import subprocess
    result = subprocess.run(
        ["tesseract", "--version"],
        capture_output=True, text=True
    )
    version = result.stdout.strip().split('\n')[0]
    result = subprocess.run(
        ["tesseract", "--list-langs"],
        capture_output=True, text=True
    )
    langs = result.stdout.strip().split('\n')[1:]
    OCR_ENGINES["tesseract"] = {
        "model": "system",
        "status": "✅ Ready",
        "languages": langs,
        "type": "System OCR engine",
        "version": version,
    }
    print(f"   ✅ {version}")
    print(f"   📋 Languages: {', '.join(langs[:10])}")
except Exception as e:
    OCR_ENGINES["tesseract"] = {"model": None, "status": f"❌ Error: {e}"}
    print(f"   ❌ Tesseract failed: {e}")

# ─── Create Test Image ───────────────────────────────────────────────
print("\n🎨 Creating test image for OCR validation...")
test_img = Image.new('RGB', (600, 200), color='white')
draw = ImageDraw.Draw(test_img)

# Draw text on image
test_text_en = "Hello World - OmniFile AI Processor v4.0"
test_text_ar = "مرحبا بالعالم - معالج ملفات أومني"

# Use default font
try:
    font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 24)
except (OSError, IOError):
    font = ImageFont.load_default()

draw.text((20, 40), test_text_en, fill='black', font=font)
draw.text((20, 100), test_text_ar, fill='black', font=font)
draw.rectangle([10, 10, 590, 190], outline='blue', width=2)

test_img_path = DIRS["processed"] / "test_image.png"
test_img.save(str(test_img_path))
print(f"   ✅ Test image saved → {test_img_path}")

# ─── Run OCR on Test Image ────────────────────────────────────────────
print("\n🧪 Testing OCR engines on sample image...")

if OCR_ENGINES.get("easyocr", {}).get("model"):
    try:
        results = easy_reader.readtext(str(test_img_path))
        print("\n   📌 EasyOCR Results:")
        for (bbox, text, conf) in results:
            print(f"      ✏️  {text}  (confidence: {conf:.2%})")
    except Exception as e:
        print(f"   ❌ EasyOCR test failed: {e}")

if OCR_ENGINES.get("trocr", {}).get("model"):
    try:
        from PIL import Image
        pixel_values = trocr_processor(test_img, return_tensors="pt").pixel_values.to(device)
        generated_ids = trocr_model.generate(pixel_values)
        generated_text = trocr_processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        print(f"\n   📌 TrOCR Result:")
        print(f"      ✏️  {generated_text}")
    except Exception as e:
        print(f"   ❌ TrOCR test failed: {e}")

# ─── Display Engine Status Summary ────────────────────────────────────
print("\n" + "=" * 70)
print("  📊 OCR Engine Status Summary")
print("=" * 70)
for name, info in OCR_ENGINES.items():
    status = info.get("status", "Unknown")
    ocr_type = info.get("type", "")
    load_time = info.get("load_time", "")
    print(f"\n   🔹 {name.upper()}")
    print(f"      Status:  {status}")
    if ocr_type:
        print(f"      Type:    {ocr_type}")
    if load_time:
        print(f"      Load:    {load_time}")

print("\n" + "=" * 70)
print("  ✅ OCR engines initialized!\n")

# Cleanup
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()


In [ ]:
# %% [markdown]
# ## 4️⃣ PDF Processing — `python notes.pdf`
# Extract text from PDF using PyMuPDF with progress tracking.

# %%
import fitz  # PyMuPDF
import time
import json
from datetime import datetime
from tqdm.notebook import tqdm
from langdetect import detect, LangDetectException

print("=" * 70)
print("  📄 PDF Processing — python notes.pdf")
print("=" * 70)

# ─── Configuration ────────────────────────────────────────────────────
PDF_PATH = "/content/drive/MyDrive/python notes.pdf"
OUTPUT_DIR = DIRS["processed"]

def process_pdf(pdf_path, output_dir):
    """Extract text from a PDF file with full metadata and progress."""

    # Verify file exists
    pdf_file = Path(pdf_path)
    if not pdf_file.exists():
        print(f"   ❌ File not found: {pdf_path}")
        print(f"   💡 Place your PDF in Google Drive at: {pdf_path}")
        print(f"   💡 Or update the PDF_PATH variable above.")
        return None

    file_size_mb = pdf_file.stat().st_size / 1e6
    print(f"\n   📁 File: {pdf_file.name}")
    print(f"   📦 Size: {file_size_mb:.2f} MB")

    # Open PDF
    doc = fitz.open(str(pdf_path))
    total_pages = len(doc)
    print(f"   📑 Pages: {total_pages}")

    # Extract text from each page
    start_time = time.time()
    all_text = []
    page_results = []

    print(f"\n   ⏳ Processing pages...")
    for page_num in tqdm(range(total_pages), desc="Extracting text"):
        page = doc[page_num]
        page_text = page.get_text("text")

        page_info = {
            "page_number": page_num + 1,
            "text": page_text.strip(),
            "char_count": len(page_text.strip()),
            "word_count": len(page_text.split()),
        }

        # Detect language
        if page_text.strip():
            try:
                lang = detect(page_text[:500])
                page_info["language"] = lang
            except LangDetectException:
                page_info["language"] = "unknown"

        all_text.append(page_text)
        page_results.append(page_info)

    elapsed = time.time() - start_time

    # Combine all text
    full_text = "\n".join(all_text).strip()
    total_words = len(full_text.split())
    total_chars = len(full_text)

    # Detect overall language
    detected_lang = "unknown"
    if full_text:
        try:
            detected_lang = detect(full_text[:2000])
        except LangDetectException:
            pass

    # ─── Display Results ────────────────────────────────────────────
    print(f"\n{'─' * 50}")
    print(f"  📊 PDF Processing Results")
    print(f"{'─' * 50}")
    print(f"   📑 Total Pages:      {total_pages}")
    print(f"   📝 Total Words:       {total_words:,}")
    print(f"   🔤 Total Characters:  {total_chars:,}")
    print(f"   🌐 Detected Language: {detected_lang}")
    print(f"   ⏱️  Processing Time:   {elapsed:.2f}s")
    print(f"   📄 Speed:             {total_pages/elapsed:.1f} pages/s")
    print(f"{'─' * 50}")

    # Show first 500 characters
    print(f"\n   📋 Text Preview (first 500 chars):\n")
    preview = full_text[:500].replace("\n", " ")
    print(f"   {preview}...\n")

    # Per-page statistics
    print(f"   📊 Per-Page Statistics:")
    for pr in page_results[:10]:  # Show first 10 pages
        lang_flag = pr.get("language", "?")
        print(f"      Page {pr['page_number']:>3d}: "
              f"{pr['word_count']:>5d} words, "
              f"{pr['char_count']:>6d} chars, "
              f"lang={lang_flag}")
    if total_pages > 10:
        print(f"      ... ({total_pages - 10} more pages)")

    # ─── Save Results ──────────────────────────────────────────────
    result_data = {
        "file": pdf_file.name,
        "file_size_mb": round(file_size_mb, 2),
        "total_pages": total_pages,
        "total_words": total_words,
        "total_characters": total_chars,
        "detected_language": detected_lang,
        "processing_time_seconds": round(elapsed, 2),
        "processed_at": datetime.now().isoformat(),
        "pages": page_results,
    }

    # Save JSON metadata
    json_path = output_dir / f"{pdf_file.stem}_metadata.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(result_data, f, ensure_ascii=False, indent=2)

    # Save extracted text
    txt_path = output_dir / f"{pdf_file.stem}_extracted.txt"
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(full_text)

    print(f"\n   💾 Results saved:")
    print(f"      📄 Text  → {txt_path}")
    print(f"      📋 Meta  → {json_path}")

    doc.close()

    return {
        "text": full_text,
        "metadata": result_data,
        "txt_path": str(txt_path),
        "json_path": str(json_path),
    }


# ─── Run PDF Processing ───────────────────────────────────────────────
pdf_result = process_pdf(PDF_PATH, OUTPUT_DIR)

if pdf_result:
    print(f"\n{'=' * 70}")
    print(f"  ✅ PDF processing complete!")
    print(f"{'=' * 70}\n")
else:
    print(f"\n   ⚠️  No file processed. Please provide a valid PDF path.\n")


In [ ]:
# %% [markdown]
# ## 5️⃣ OCR on Images
# Upload images or select from Drive. Process with EasyOCR and TrOCR.

# %%
from PIL import Image
import io
import glob
from IPython.display import display, HTML

print("=" * 70)
print("  🖼️  Image OCR Processing")
print("=" * 70)

# ─── Method 1: Upload Image ───────────────────────────────────────────
print("\n📤 Upload an image file:")
from google.colab import files

uploaded = files.upload()
uploaded_images = list(uploaded.keys())

if uploaded_images:
    print(f"   ✅ Uploaded {len(uploaded_images)} image(s): {uploaded_images}")
else:
    print("   ℹ️  No images uploaded. Checking Drive for images...")
    drive_images = list(DIRS["raw_images"].glob("*.*"))
    uploaded_images = [str(img) for img in drive_images]
    if drive_images:
        print(f"   ✅ Found {len(drive_images)} image(s) in Drive")


def process_image_ocr(image_path, engines=None):
    """
    Process an image with multiple OCR engines.

    Args:
        image_path: Path to the image file
        engines: List of engines to use ('easyocr', 'trocr')

    Returns:
        dict with results from each engine
    """
    if engines is None:
        engines = ["easyocr", "trocr"]

    results = {}
    img = Image.open(image_path)
    print(f"\n   🖼️  Processing: {Path(image_path).name}")
    print(f"      Size: {img.size[0]}×{img.size[1]}")

    # Display image
    display(img)

    # ─── EasyOCR ──────────────────────────────────────────────────
    if "easyocr" in engines and OCR_ENGINES.get("easyocr", {}).get("model"):
        try:
            reader = OCR_ENGINES["easyocr"]["model"]
            start_time = time.time()
            ocr_results = reader.readtext(str(image_path))
            elapsed = time.time() - start_time

            texts = []
            confidences = []
            for (bbox, text, conf) in ocr_results:
                texts.append(text)
                confidences.append(conf)
                print(f"      ✏️  [{conf:.0%}] {text}")

            avg_conf = sum(confidences) / len(confidences) if confidences else 0
            results["easyocr"] = {
                "texts": texts,
                "full_text": "\n".join(texts),
                "avg_confidence": avg_conf,
                "num_detections": len(ocr_results),
                "processing_time": elapsed,
            }
            print(f"      📊 EasyOCR: {len(ocr_results)} detections, "
                  f"avg confidence: {avg_conf:.1%}, "
                  f"time: {elapsed:.2f}s")
        except Exception as e:
            print(f"      ❌ EasyOCR error: {e}")
            results["easyocr"] = {"error": str(e)}

    # ─── TrOCR ───────────────────────────────────────────────────
    if "trocr" in engines and OCR_ENGINES.get("trocr", {}).get("model"):
        try:
            trocr_processor = OCR_ENGINES["trocr"]["processor"]
            trocr_model = OCR_ENGINES["trocr"]["model"]

            start_time = time.time()
            pixel_values = trocr_processor(img, return_tensors="pt").pixel_values.to(device)
            generated_ids = trocr_model.generate(pixel_values)
            generated_text = trocr_processor.batch_decode(
                generated_ids, skip_special_tokens=True
            )[0]
            elapsed = time.time() - start_time

            results["trocr"] = {
                "full_text": generated_text,
                "processing_time": elapsed,
            }
            print(f"      📊 TrOCR: \"{generated_text}\" (time: {elapsed:.2f}s)")
        except Exception as e:
            print(f"      ❌ TrOCR error: {e}")
            results["trocr"] = {"error": str(e)}

    return results


# ─── Process Uploaded Images ──────────────────────────────────────────
all_ocr_results = {}

for img_name in uploaded_images:
    print(f"\n{'─' * 50}")
    ocr_results = process_image_ocr(img_name)
    all_ocr_results[img_name] = ocr_results

    # Save results
    if ocr_results:
        output_path = save_to_drive(
            ocr_results,
            f"{Path(img_name).stem}_ocr_results.json",
            subdir="processed"
        )

# ─── Batch Processing from Drive ──────────────────────────────────────
print(f"\n{'─' * 50}")
print("   📦 Batch Processing — Drive Images")

drive_image_files = list(DIRS["raw_images"].glob("*.*"))
SUPPORTED_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tiff", ".webp"}
drive_image_files = [f for f in drive_image_files if f.suffix.lower() in SUPPORTED_EXTS]

if drive_image_files:
    print(f"   Found {len(drive_image_files)} image(s) in Drive/raw_images/")
    batch_results = {}

    for img_path in drive_image_files:
        print(f"\n   🖼️  Processing: {img_path.name}")
        result = process_image_ocr(str(img_path))
        batch_results[img_path.name] = result

    # Save batch results
    save_to_drive(batch_results, "batch_ocr_results.json", subdir="processed")
    print(f"\n   ✅ Batch processing complete: {len(batch_results)} image(s)")
else:
    print(f"   ℹ️  No images found in {DIRS['raw_images']}")
    print(f"      Add images to your Drive: {DIRS['raw_images']}")

print(f"\n{'=' * 70}")
print(f"  ✅ Image OCR complete!")
print(f"{'=' * 70}\n")


In [ ]:
# %% [markdown]
# ## 6️⃣ Spell Correction
# Arabic correction (ar-corrector) and English correction (pyspellchecker).

# %%
print("=" * 70)
print("  ✏️  Spell Correction Engine")
print("=" * 70)

# ─── 1. Arabic Spell Correction ────────────────────────────────────────
print("\n🔤 Arabic Spell Correction (ar-corrector)")

try:
    from ar_corrector import ArCorrector
    ar_corrector = ArCorrector()
    print("   ✅ ar-corrector loaded")
except ImportError:
    print("   ⚠️  ar-corrector not available. Installing...")
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ar-corrector"])
    from ar_corrector import ArCorrector
    ar_corrector = ArCorrector()
    print("   ✅ ar-corrector installed and loaded")

arabic_test_sentences = [
    "هذا النص يحتوي على اخطاء املية يجب تصحيحها",
    "التعليم مهم جدا للنجاح في الحياة المهنية",
    "الذكاء الاصطناعي يغير العالم بشكل جذري",
    "هناك كثير من التحديات في عالم التقنية",
    "البحث العلمي يساهم في تقدم المجتمعات",
]

print("\n   📝 Arabic Correction Results:")
print("   " + "─" * 55)
arabic_results = []

for i, sentence in enumerate(arabic_test_sentences, 1):
    try:
        corrected = ar_corrector.correct_text(sentence)
        changed = sentence != corrected
        status = "🔧" if changed else "✅"

        print(f"\n   [{i}] {status}")
        print(f"   Before: {sentence}")
        print(f"   After:  {corrected}")

        arabic_results.append({
            "original": sentence,
            "corrected": corrected,
            "changed": changed,
        })
    except Exception as e:
        print(f"   [{i}] ❌ Error: {e}")

# ─── 2. English Spell Correction ───────────────────────────────────────
print("\n\n🔤 English Spell Correction (pyspellchecker)")

try:
    from spellchecker import SpellChecker
    spell = SpellChecker()
    print("   ✅ pyspellchecker loaded")
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyspellchecker"])
    from spellchecker import SpellChecker
    spell = SpellChecker()
    print("   ✅ pyspellchecker installed and loaded")


def correct_english_text(text):
    """
    Correct English text word by word using pyspellchecker.
    Preserves punctuation and capitalization.
    """
    import re
    words = re.findall(r"[a-zA-Z']+|[^a-zA-Z]+", text)
    corrected_words = []
    corrections_made = []

    for word in words:
        if re.match(r"[a-zA-Z]", word):
            lower_word = word.lower()
            if lower_word in spell.unknown([lower_word]):
                candidate = spell.correction(lower_word)
                if candidate and candidate != lower_word:
                    # Preserve capitalization
                    if word[0].isupper():
                        candidate = candidate.capitalize()
                    corrections_made.append((word, candidate))
                    corrected_words.append(candidate)
                else:
                    corrected_words.append(word)
            else:
                corrected_words.append(word)
        else:
            corrected_words.append(word)

    corrected_text = "".join(corrected_words)
    return corrected_text, corrections_made


english_test_sentences = [
    "This is a test sentnce with som misspelled wrds",
    "Artifical intellegence is changin the wrld",
    "The quik brown fox jumps over the lazzy dog",
    "Optical character recogniton has improvd significntly",
    "Machine lerning algorthms process natural languge",
]

print("\n   📝 English Correction Results:")
print("   " + "─" * 55)
english_results = []

for i, sentence in enumerate(english_test_sentences, 1):
    corrected, corrections = correct_english_text(sentence)
    changed = sentence != corrected
    status = "🔧" if changed else "✅"

    print(f"\n   [{i}] {status}")
    print(f"   Before: {sentence}")
    print(f"   After:  {corrected}")
    if corrections:
        print(f"   Fixes:  {', '.join(f'{w}→{c}' for w, c in corrections)}")

    english_results.append({
        "original": sentence,
        "corrected": corrected,
        "corrections": corrections,
        "changed": changed,
    })

# ─── 3. Custom Correction Dictionary ──────────────────────────────────
print("\n\n📖 Custom Correction Dictionary")
print("   " + "─" * 55)

custom_corrections = {
    # Technical terms that spell checkers often miss
    "ocr": "OCR",
    "ai": "AI",
    "ml": "ML",
    "nlp": "NLP",
    "trocr": "TrOCR",
    "omnifile": "OmniFile",
    # Common domain-specific corrections
    "recogniton": "recognition",
    "precessing": "processing",
    "extration": "extraction",
    "transaltion": "translation",
}

print("   Custom dictionary entries:")
for wrong, correct in custom_corrections.items():
    print(f"      {wrong:>15s} → {correct}")

def apply_custom_corrections(text, dictionary):
    """Apply custom word replacements to text."""
    import re
    corrected = text
    for wrong, correct in dictionary.items():
        corrected = re.sub(rf'\b{re.escape(wrong)}\b', correct, corrected, flags=re.IGNORECASE)
    return corrected


custom_test = "The ocr system uses ai and ml for recogniton and precessing"
custom_result = apply_custom_corrections(custom_test, custom_corrections)
print(f"\n   Test:  {custom_test}")
print(f"   Result: {custom_result}")

# ─── Save Results ────────────────────────────────────────────────────
correction_results = {
    "arabic": arabic_results,
    "english": english_results,
    "custom_dictionary": custom_corrections,
}

save_to_drive(correction_results, "spell_correction_results.json", subdir="processed")

print(f"\n{'=' * 70}")
print(f"  ✅ Spell correction complete!")
print(f"{'=' * 70}\n")


In [ ]:
# %% [markdown]
# ## 7️⃣ Translation Engine
# English ↔ Arabic translation using Helsinki-NLP/opus-mt models.

# %%
print("=" * 70)
print("  🌐 Translation Engine")
print("=" * 70)

from transformers import MarianMTModel, MarianTokenizer
import torch
import gc
import time

device = "cuda" if torch.cuda.is_available() else "cpu"

# ─── 1. English → Arabic ──────────────────────────────────────────────
print("\n🔄 Loading English → Arabic model")
print("   Model: Helsinki-NLP/opus-mt-en-ar")

try:
    start_time = time.time()
    en_ar_tokenizer = MarianTokenizer.from_pretrained(
        "Helsinki-NLP/opus-mt-en-ar",
        cache_dir=str(DIRS["models_cache"]),
    )
    en_ar_model = MarianMTModel.from_pretrained(
        "Helsinki-NLP/opus-mt-en-ar",
        cache_dir=str(DIRS["models_cache"]),
    ).to(device)
    print(f"   ✅ Model loaded in {time.time() - start_time:.1f}s")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    en_ar_model = None
    en_ar_tokenizer = None


# ─── 2. Arabic → English ──────────────────────────────────────────────
print("\n🔄 Loading Arabic → English model")
print("   Model: Helsinki-NLP/opus-mt-ar-en")

try:
    start_time = time.time()
    ar_en_tokenizer = MarianTokenizer.from_pretrained(
        "Helsinki-NLP/opus-mt-ar-en",
        cache_dir=str(DIRS["models_cache"]),
    )
    ar_en_model = MarianMTModel.from_pretrained(
        "Helsinki-NLP/opus-mt-ar-en",
        cache_dir=str(DIRS["models_cache"]),
    ).to(device)
    print(f"   ✅ Model loaded in {time.time() - start_time:.1f}s")
except Exception as e:
    print(f"   ❌ Failed to load model: {e}")
    ar_en_model = None
    ar_en_tokenizer = None


# ─── Translation Functions ────────────────────────────────────────────
def translate_en_to_ar(text, model=en_ar_model, tokenizer=en_ar_tokenizer):
    """Translate English text to Arabic."""
    if model is None:
        return "❌ Model not available"
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
    translated = model.generate(**inputs, max_length=512)
    return tokenizer.decode(translated[0], skip_special_tokens=True)


def translate_ar_to_en(text, model=ar_en_model, tokenizer=ar_en_tokenizer):
    """Translate Arabic text to English."""
    if model is None:
        return "❌ Model not available"
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
    translated = model.generate(**inputs, max_length=512)
    return tokenizer.decode(translated[0], skip_special_tokens=True)


def batch_translate(texts, direction="en-ar", batch_size=8):
    """
    Batch translate a list of texts.

    Args:
        texts: List of strings to translate
        direction: 'en-ar' or 'ar-en'
        batch_size: Number of texts to process at once
    """
    if direction == "en-ar":
        model, tokenizer = en_ar_model, en_ar_tokenizer
    else:
        model, tokenizer = ar_en_model, ar_en_tokenizer

    if model is None:
        return ["❌ Model not available"] * len(texts)

    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(
            batch, return_tensors="pt", padding=True, truncation=True, max_length=512
        ).to(device)
        translated = model.generate(**inputs, max_length=512)
        decoded = tokenizer.batch_decode(translated, skip_special_tokens=True)
        results.extend(decoded)

    return results


# ─── 3. Test Translations ─────────────────────────────────────────────
print("\n" + "─" * 55)
print("   📝 Translation Tests")
print("─" * 55)

# English → Arabic
en_sentences = [
    "Artificial intelligence is transforming the world of technology.",
    "The OmniFile processor handles multiple languages for OCR.",
    "Machine learning algorithms improve accuracy over time.",
    "Document processing requires advanced natural language understanding.",
    "The future of AI lies in multimodal processing capabilities.",
]

print("\n   🇬🇧 → 🇸🇦 English to Arabic:")
en_ar_results = []

for i, sentence in enumerate(en_sentences, 1):
    start_time = time.time()
    translated = translate_en_to_ar(sentence)
    elapsed = time.time() - start_time
    print(f"\n   [{i}] ({elapsed:.2f}s)")
    print(f"   EN: {sentence}")
    print(f"   AR: {translated}")
    en_ar_results.append({"en": sentence, "ar": translated})

# Arabic → English
ar_sentences = [
    "الذكاء الاصطناعي يغير عالم التكنولوجيا بشكل كبير",
    "معالجة المستندات تتطلب فهما متقدما للغة الطبيعية",
    "التعلم الآلي يحسن الدقة مع مرور الوقت",
    "مستقبل الذكاء الاصطناعي في القدرات متعددة الوسائط",
]

print("\n\n   🇸🇦 → 🇬🇧 Arabic to English:")
ar_en_results = []

for i, sentence in enumerate(ar_sentences, 1):
    start_time = time.time()
    translated = translate_ar_to_en(sentence)
    elapsed = time.time() - start_time
    print(f"\n   [{i}] ({elapsed:.2f}s)")
    print(f"   AR: {sentence}")
    print(f"   EN: {translated}")
    ar_en_results.append({"ar": sentence, "en": translated})

# ─── 4. Batch Translation Demo ────────────────────────────────────────
print("\n\n📦 Batch Translation Demo")
print("─" * 55)

batch_en = [
    "Hello, how are you?",
    "The weather is nice today.",
    "I love programming.",
    "Data science is fascinating.",
]

print(f"\n   Translating {len(batch_en)} sentences (EN→AR) in batch...")
start_time = time.time()
batch_results = batch_translate(batch_en, "en-ar")
elapsed = time.time() - start_time

for original, translated in zip(batch_en, batch_results):
    print(f"   EN: {original}")
    print(f"   AR: {translated}")
    print()

print(f"   ⏱️  Batch time: {elapsed:.2f}s for {len(batch_en)} sentences")

# ─── Save Results ────────────────────────────────────────────────────
translation_results = {
    "en_to_ar": en_ar_results,
    "ar_to_en": ar_en_results,
    "batch_demo": list(zip(batch_en, batch_results)),
}

save_to_drive(translation_results, "translation_results.json", subdir="processed")

# Cleanup
gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

print(f"\n{'=' * 70}")
print(f"  ✅ Translation complete!")
print(f"{'=' * 70}\n")


In [ ]:
# %% [markdown]
# ## 8️⃣ Evaluation Metrics (CER / WER)
# Calculate Character Error Rate and Word Error Rate.
# Compare OCR output against reference text with visualization.

# %%
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

print("=" * 70)
print("  📊 Evaluation Metrics — CER & WER")
print("=" * 70)


# ─── 1. Metric Calculation Functions ──────────────────────────────────
def calculate_cer(reference, hypothesis):
    """
    Calculate Character Error Rate (CER).
    CER = (S + D + I) / N
    where S = substitutions, D = deletions, I = insertions, N = reference length
    """
    import difflib

    ref = list(reference.replace(" ", ""))
    hyp = list(hypothesis.replace(" ", ""))

    if not ref:
        return 1.0 if hyp else 0.0

    # Use Levenshtein distance via difflib
    matcher = difflib.SequenceMatcher(None, ref, hyp)

    # Calculate edit distance
    m, n = len(ref), len(hyp)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if ref[i - 1] == hyp[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])

    edit_distance = dp[m][n]
    cer = edit_distance / m
    return cer


def calculate_wer(reference, hypothesis):
    """
    Calculate Word Error Rate (WER).
    WER = (S + D + I) / N
    where N = number of words in reference
    """
    ref_words = reference.split()
    hyp_words = hypothesis.split()

    if not ref_words:
        return 1.0 if hyp_words else 0.0

    m, n = len(ref_words), len(hyp_words)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if ref_words[i - 1] == hyp_words[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])

    edit_distance = dp[m][n]
    wer = edit_distance / m
    return wer


def get_error_details(reference, hypothesis):
    """Get detailed error analysis (substitutions, deletions, insertions)."""
    ref_words = reference.split()
    hyp_words = hypothesis.split()

    if not ref_words:
        return {"substitutions": 0, "deletions": 0, "insertions": 0, "correct": 0}

    m, n = len(ref_words), len(hyp_words)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    ops = [[[] for _ in range(n + 1)] for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if ref_words[i - 1] == hyp_words[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(dp[i - 1][j], dp[i][j - 1], dp[i - 1][j - 1])

    # Backtrack to count error types
    i, j = m, n
    substitutions = 0
    deletions = 0
    insertions = 0
    correct = 0

    while i > 0 or j > 0:
        if i > 0 and j > 0 and ref_words[i - 1] == hyp_words[j - 1]:
            correct += 1
            i -= 1
            j -= 1
        elif j > 0 and (i == 0 or dp[i][j - 1] <= dp[i - 1][j]):
            insertions += 1
            j -= 1
        elif i > 0:
            if j > 0 and dp[i][j] == dp[i - 1][j - 1] + 1:
                substitutions += 1
                j -= 1
            else:
                deletions += 1
            i -= 1

    return {
        "substitutions": substitutions,
        "deletions": deletions,
        "insertions": insertions,
        "correct": correct,
    }


# ─── 2. Evaluation Test Cases ─────────────────────────────────────────
print("\n📝 Evaluation Test Cases")
print("─" * 55)

test_pairs = [
    {
        "name": "English - Clean",
        "reference": "The quick brown fox jumps over the lazy dog",
        "hypothesis": "The quick brown fox jumps over the lazy dog",
    },
    {
        "name": "English - Minor Errors",
        "reference": "Artificial intelligence is transforming technology",
        "hypothesis": "Artificial inteligence is transformng technology",
    },
    {
        "name": "English - OCR Errors",
        "reference": "Document processing and optical character recognition",
        "hypothesis": "Docunent processing and optical character recogniton",
    },
    {
        "name": "English - Missing Words",
        "reference": "The conference on machine learning was held in Paris",
        "hypothesis": "The conference on machine was held in Paris",
    },
    {
        "name": "English - Extra Words",
        "reference": "Natural language processing techniques",
        "hypothesis": "the natural language processing techniques and methods",
    },
    {
        "name": "Arabic - Clean",
        "reference": "الذكاء الاصطناعي يغير العالم",
        "hypothesis": "الذكاء الاصطناعي يغير العالم",
    },
    {
        "name": "Arabic - OCR Errors",
        "reference": "معالجة اللغة الطبيعية",
        "hypothesis": "معالجة اللغة الطبيعية",
    },
]

# ─── 3. Calculate Metrics ─────────────────────────────────────────────
results_data = []

print(f"\n{'Name':<25} {'CER':>8} {'WER':>8} {'S':>4} {'D':>4} {'I':>4} {'C':>4}")
print("─" * 70)

for pair in test_pairs:
    cer = calculate_cer(pair["reference"], pair["hypothesis"])
    wer = calculate_wer(pair["reference"], pair["hypothesis"])
    details = get_error_details(pair["reference"], pair["hypothesis"])

    results_data.append({
        "name": pair["name"],
        "cer": cer,
        "wer": wer,
        **details,
    })

    print(f"{pair['name']:<25} {cer:>8.4f} {wer:>8.4f} "
          f"{details['substitutions']:>4} {details['deletions']:>4} "
          f"{details['insertions']:>4} {details['correct']:>4}")

# ─── 4. Visualization ────────────────────────────────────────────────
print("\n📊 Generating visualizations...")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('OmniFile AI Processor — OCR Evaluation Metrics', fontsize=14, fontweight='bold')

names = [r["name"] for r in results_data]
cers = [r["cer"] * 100 for r in results_data]
wers = [r["wer"] * 100 for r in results_data]

# ─── CER Bar Chart ────────────────────────────────────────────────
ax1 = axes[0, 0]
colors_cer = ['#22c55e' if c == 0 else '#ef4444' if c > 10 else '#f97316' for c in cers]
bars1 = ax1.barh(names, cers, color=colors_cer, alpha=0.8, edgecolor='white')
ax1.set_xlabel('Character Error Rate (%)')
ax1.set_title('CER by Test Case', fontweight='bold')
ax1.set_xlim(0, max(cers) * 1.2 + 1)
for bar, val in zip(bars1, cers):
    ax1.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
             f'{val:.1f}%', va='center', fontsize=9)
ax1.invert_yaxis()

# ─── WER Bar Chart ────────────────────────────────────────────────
ax2 = axes[0, 1]
colors_wer = ['#22c55e' if w == 0 else '#ef4444' if w > 15 else '#f97316' for w in wers]
bars2 = ax2.barh(names, wers, color=colors_wer, alpha=0.8, edgecolor='white')
ax2.set_xlabel('Word Error Rate (%)')
ax2.set_title('WER by Test Case', fontweight='bold')
ax2.set_xlim(0, max(wers) * 1.2 + 1)
for bar, val in zip(bars2, wers):
    ax2.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
             f'{val:.1f}%', va='center', fontsize=9)
ax2.invert_yaxis()

# ─── Error Type Breakdown ────────────────────────────────────────
ax3 = axes[1, 0]
total_sub = sum(r["substitutions"] for r in results_data)
total_del = sum(r["deletions"] for r in results_data)
total_ins = sum(r["insertions"] for r in results_data)
total_cor = sum(r["correct"] for r in results_data)

error_labels = ['Substitutions', 'Deletions', 'Insertions', 'Correct']
error_values = [total_sub, total_del, total_ins, total_cor]
error_colors = ['#ef4444', '#f97316', '#eab308', '#22c55e']
explode = (0.05, 0.05, 0.05, 0.1)

wedges, texts, autotexts = ax3.pie(
    error_values, labels=error_labels, colors=error_colors,
    autopct='%1.1f%%', startangle=90, explode=explode
)
ax3.set_title('Error Type Distribution', fontweight='bold')

# ─── CER vs WER Scatter ────────────────────────────────────────
ax4 = axes[1, 1]
scatter = ax4.scatter(cers, wers, c=range(len(cers)), cmap='viridis', s=120, edgecolors='white', zorder=5)
ax4.plot([0, max(max(cers), max(wers))], [0, max(max(cers), max(wers))],
         'r--', alpha=0.5, label='CER = WER line')
ax4.set_xlabel('CER (%)')
ax4.set_ylabel('WER (%)')
ax4.set_title('CER vs WER Comparison', fontweight='bold')
ax4.legend(fontsize=8)

# Add labels to points
for i, name in enumerate(names):
    ax4.annotate(name, (cers[i], wers[i]), textcoords="offset points",
                 xytext=(5, 5), fontsize=7)

plt.tight_layout()
plt.savefig(str(DIRS["processed"] / "evaluation_metrics.png"), dpi=150, bbox_inches='tight')
plt.show()
print(f"   💾 Chart saved → {DIRS['processed'] / 'evaluation_metrics.png'}")

# ─── 5. Summary Statistics ───────────────────────────────────────────
avg_cer = np.mean(cers)
avg_wer = np.mean(wers)
best_cer = min(cers)
best_wer = min(wers)
worst_cer = max(cers)
worst_wer = max(wers)

print(f"\n{'─' * 55}")
print(f"   📊 Summary Statistics")
print(f"{'─' * 55}")
print(f"   Average CER:   {avg_cer:.2f}%")
print(f"   Average WER:   {avg_wer:.2f}%")
print(f"   Best CER:      {best_cer:.2f}% ({names[cers.index(best_cer)]})")
print(f"   Best WER:      {best_wer:.2f}% ({names[wers.index(best_wer)]})")
print(f"   Worst CER:     {worst_cer:.2f}% ({names[cers.index(worst_cer)]})")
print(f"   Worst WER:     {worst_wer:.2f}% ({names[wers.index(worst_wer)]})")
print(f"{'─' * 55}")

# Save evaluation results
save_to_drive({
    "test_pairs": test_pairs,
    "results": results_data,
    "summary": {
        "avg_cer": avg_cer,
        "avg_wer": avg_wer,
        "best_cer": best_cer,
        "best_wer": best_wer,
    },
}, "evaluation_results.json", subdir="processed")

print(f"\n{'=' * 70}")
print(f"  ✅ Evaluation complete!")
print(f"{'=' * 70}\n")


In [ ]:
# %% [markdown]
# ## 9️⃣ Launch Gradio Web UI
# Build and launch a full-featured Gradio app with 7 tabs and ngrok tunnel.

# %%
import gradio as gr
import torch
import numpy as np
from PIL import Image
import time
import gc
import json

print("=" * 70)
print("  🖥️  Building Gradio Web Interface")
print("=" * 70)

device = "cuda" if torch.cuda.is_available() else "cpu"


# ═══════════════════════════════════════════════════════════════════
# Tab 1: PDF Processing
# ═══════════════════════════════════════════════════════════════════

def process_pdf_tab(pdf_file):
    """Process uploaded PDF and return extracted text."""
    if pdf_file is None:
        return "Please upload a PDF file.", "No file provided."

    try:
        import fitz
        start_time = time.time()
        doc = fitz.open(pdf_file.name)
        total_pages = len(doc)

        all_text = []
        for page in doc:
            all_text.append(page.get_text("text"))

        full_text = "\n".join(all_text).strip()
        elapsed = time.time() - start_time
        doc.close()

        summary = (
            f"📄 **File:** {pdf_file.name}\n"
            f"📑 **Pages:** {total_pages}\n"
            f"📝 **Words:** {len(full_text.split()):,}\n"
            f"🔤 **Characters:** {len(full_text):,}\n"
            f"⏱️ **Time:** {elapsed:.2f}s"
        )
        return full_text, summary
    except Exception as e:
        return f"Error: {str(e)}", f"Processing failed: {e}"


# ═══════════════════════════════════════════════════════════════════
# Tab 2: Image OCR (EasyOCR)
# ═══════════════════════════════════════════════════════════════════

def ocr_easyocr_tab(image, languages):
    """Process image with EasyOCR."""
    if image is None:
        return "Please upload an image.", "No image provided."

    try:
        lang_map = {
            "Arabic": "ar",
            "English": "en",
            "German": "de",
            "French": "fr",
        }
        selected_langs = [lang_map.get(l, "en") for l in languages if l in lang_map]

        import easyocr
        reader = easyocr.Reader(selected_langs, gpu=(device == "cuda"))

        start_time = time.time()
        results = reader.readtext(np.array(image))
        elapsed = time.time() - start_time

        texts = []
        confidences = []
        output_lines = []

        for bbox, text, conf in results:
            texts.append(text)
            confidences.append(conf)
            output_lines.append(f"[{conf:.0%}] {text}")

        full_text = "\n".join(texts)
        avg_conf = np.mean(confidences) * 100 if confidences else 0

        details = (
            f"🔍 **Engine:** EasyOCR\n"
            f"🌐 **Languages:** {', '.join(languages)}\n"
            f"📝 **Detections:** {len(results)}\n"
            f"📊 **Avg Confidence:** {avg_conf:.1f}%\n"
            f"⏱️ **Time:** {elapsed:.2f}s"
        )
        return full_text, f"\n".join(output_lines), details
    except Exception as e:
        return f"Error: {str(e)}", "", f"Processing failed: {e}"


# ═══════════════════════════════════════════════════════════════════
# Tab 3: Handwriting OCR (TrOCR)
# ═══════════════════════════════════════════════════════════════════

def ocr_trocr_tab(image):
    """Process handwritten image with TrOCR."""
    if image is None:
        return "Please upload an image.", "No image provided."

    try:
        from transformers import TrOCRProcessor, VisionEncoderDecoderModel

        processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
        model = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten").to(device)

        start_time = time.time()
        pixel_values = processor(image, return_tensors="pt").pixel_values.to(device)
        generated_ids = model.generate(pixel_values)
        generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
        elapsed = time.time() - start_time

        del model
        del processor
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()

        details = (
            f"🔍 **Engine:** TrOCR\n"
            f"📝 **Model:** microsoft/trocr-base-handwritten\n"
            f"🔤 **Text:** {generated_text}\n"
            f"⏱️ **Time:** {elapsed:.2f}s"
        )
        return generated_text, details
    except Exception as e:
        return f"Error: {str(e)}", f"Processing failed: {e}"


# ═══════════════════════════════════════════════════════════════════
# Tab 4: Spell Correction
# ═══════════════════════════════════════════════════════════════════

def correct_spelling_tab(text, language):
    """Correct spelling in the given text."""
    if not text or not text.strip():
        return "Please enter some text.", "No text provided."

    try:
        start_time = time.time()

        if language == "Arabic":
            from ar_corrector import ArCorrector
            corrector = ArCorrector()
            corrected = corrector.correct_text(text)
            engine = "ar-corrector"
        else:
            from spellchecker import SpellChecker
            spell = SpellChecker()
            import re
            words = re.findall(r"[a-zA-Z']+|[^a-zA-Z]+", text)
            corrected_words = []
            corrections = []
            for w in words:
                if re.match(r"[a-zA-Z]", w):
                    lower = w.lower()
                    if lower in spell.unknown([lower]):
                        c = spell.correction(lower)
                        if c and c != lower:
                            corrections.append(f"{w}→{c}")
                            if w[0].isupper():
                                c = c.capitalize()
                            corrected_words.append(c)
                            continue
                corrected_words.append(w)
            corrected = "".join(corrected_words)
            engine = "pyspellchecker"

        elapsed = time.time() - start_time

        diff = "\n".join([f"   🔧 {text}", f"   ✅ {corrected}"])
        details = (
            f"✏️ **Engine:** {engine}\n"
            f"🌐 **Language:** {language}\n"
            f"⏱️ **Time:** {elapsed:.2f}s\n"
            f"📝 **Changed:** {text != corrected}"
        )
        return corrected, details
    except Exception as e:
        return f"Error: {str(e)}", f"Correction failed: {e}"


# ═══════════════════════════════════════════════════════════════════
# Tab 5: Translation
# ═══════════════════════════════════════════════════════════════════

def translate_tab(text, direction):
    """Translate text between English and Arabic."""
    if not text or not text.strip():
        return "Please enter some text.", "No text provided."

    try:
        from transformers import MarianMTModel, MarianTokenizer

        model_name = (
            "Helsinki-NLP/opus-mt-en-ar" if direction == "English → Arabic"
            else "Helsinki-NLP/opus-mt-ar-en"
        )

        tokenizer = MarianTokenizer.from_pretrained(model_name)
        model = MarianMTModel.from_pretrained(model_name).to(device)

        start_time = time.time()
        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)
        translated = model.generate(**inputs, max_length=512)
        result = tokenizer.decode(translated[0], skip_special_tokens=True)
        elapsed = time.time() - start_time

        del model
        del tokenizer
        gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()

        details = (
            f"🌐 **Direction:** {direction}\n"
            f"📝 **Model:** {model_name}\n"
            f"⏱️ **Time:** {elapsed:.2f}s"
        )
        return result, details
    except Exception as e:
        return f"Error: {str(e)}", f"Translation failed: {e}"


# ═══════════════════════════════════════════════════════════════════
# Tab 6: Evaluation Metrics
# ═══════════════════════════════════════════════════════════════════

def evaluate_tab(reference, hypothesis):
    """Calculate CER and WER metrics."""
    if not reference or not hypothesis:
        return "Please enter both reference and hypothesis.", "No text provided."

    import difflib

    # CER
    ref_chars = list(reference.replace(" ", ""))
    hyp_chars = list(hypothesis.replace(" ", ""))
    m, n = len(ref_chars), len(hyp_chars)
    if m == 0:
        cer = 1.0
    else:
        dp = [[0]*(n+1) for _ in range(m+1)]
        for i in range(m+1): dp[i][0] = i
        for j in range(n+1): dp[0][j] = j
        for i in range(1, m+1):
            for j in range(1, n+1):
                if ref_chars[i-1] == hyp_chars[j-1]:
                    dp[i][j] = dp[i-1][j-1]
                else:
                    dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
        cer = dp[m][n] / m

    # WER
    ref_words = reference.split()
    hyp_words = hypothesis.split()
    mw, nw = len(ref_words), len(hyp_words)
    if mw == 0:
        wer = 1.0
    else:
        dp = [[0]*(nw+1) for _ in range(mw+1)]
        for i in range(mw+1): dp[i][0] = i
        for j in range(nw+1): dp[0][j] = j
        for i in range(1, mw+1):
            for j in range(1, nw+1):
                if ref_words[i-1] == hyp_words[j-1]:
                    dp[i][j] = dp[i-1][j-1]
                else:
                    dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
        wer = dp[mw][nw] / mw

    accuracy = max(0, (1 - wer) * 100)

    # Diff
    diff = difflib.unified_diff(
        reference.splitlines(keepends=True),
        hypothesis.splitlines(keepends=True),
        fromfile="Reference", tofile="Hypothesis", lineterm=""
    )
    diff_text = "\n".join(diff)

    results = (
        f"📊 **Character Error Rate (CER):** {cer:.4f} ({cer*100:.2f}%)\n\n"
        f"📊 **Word Error Rate (WER):** {wer:.4f} ({wer*100:.2f}%)\n\n"
        f"✅ **Accuracy:** {accuracy:.2f}%\n\n"
        f"📝 **Reference words:** {mw}\n"
        f"📝 **Hypothesis words:** {nw}\n"
        f"📝 **Reference chars:** {m}\n"
        f"📝 **Hypothesis chars:** {n}"
    )

    return results, diff_text or "No differences."


# ═══════════════════════════════════════════════════════════════════
# Tab 7: About
# ═══════════════════════════════════════════════════════════════════

about_content = """
# 📄 OmniFile AI Processor v4.0

### Multilingual Document Intelligence Platform

## 👨‍💻 Author
**Dr. Abdulmalek**

## 🔗 Links
- **GitHub:** [OmniFile_Processor](https://github.com/DrAbdulmalek/OmniFile_Processor)
- **HuggingFace:** [handwriting-ocr Space](https://huggingface.co/spaces/DrAbdulmalek/handwriting-ocr)

## ✨ Features
- 📄 PDF text extraction (PyMuPDF)
- 🖼️ Multi-language OCR (EasyOCR)
- ✍️ Handwriting recognition (TrOCR)
- ✏️ Spell correction (Arabic & English)
- 🌐 Translation (English ↔ Arabic)
- 📊 Evaluation metrics (CER / WER)

## 🛠️ Tech Stack
- EasyOCR, TrOCR (Microsoft)
- Helsinki-NLP MarianMT
- PyMuPDF, ar-corrector, pyspellchecker
- Gradio, PyTorch
"""


# ═══════════════════════════════════════════════════════════════════
# Build Gradio Interface
# ═══════════════════════════════════════════════════════════════════

print("\n🏗️  Building Gradio interface with 7 tabs...")

with gr.Blocks(
    title="OmniFile AI Processor v4.0",
    theme=gr.themes.Soft(),
    css="""
        .gradio-container { max-width: 1200px !important; }
        footer { display: none !important; }
    """
) as demo:

    gr.Markdown(
        "# 📄 OmniFile AI Processor v4.0\n"
        "### Multilingual OCR · Translation · Spell Correction\n"
        "**by Dr. Abdulmalek**"
    )

    with gr.Tabs():

        # ─── Tab 1: PDF Processing ──────────────────────────────────
        with gr.Tab("📄 PDF Processing"):
            gr.Markdown("### Upload a PDF to extract text")
            with gr.Row():
                with gr.Column():
                    pdf_input = gr.File(label="Upload PDF", file_types=[".pdf"])
                    pdf_btn = gr.Button("Extract Text", variant="primary")
                with gr.Column():
                    pdf_summary = gr.Markdown(label="Summary")
            pdf_output = gr.Textbox(label="Extracted Text", lines=15, show_copy_button=True)
            pdf_btn.click(fn=process_pdf_tab, inputs=[pdf_input], outputs=[pdf_output, pdf_summary])

        # ─── Tab 2: Image OCR ──────────────────────────────────────
        with gr.Tab("🖼️ Image OCR"):
            gr.Markdown("### Upload an image for OCR")
            with gr.Row():
                with gr.Column():
                    img_input = gr.Image(label="Upload Image", type="pil")
                    img_langs = gr.CheckboxGroup(
                        choices=["Arabic", "English", "German", "French"],
                        value=["English", "Arabic"],
                        label="Languages"
                    )
                    img_btn = gr.Button("Run OCR", variant="primary")
                with gr.Column():
                    img_details = gr.Markdown(label="Details")
            img_detections = gr.Textbox(label="Detections (with confidence)", lines=10)
            img_output = gr.Textbox(label="Extracted Text", lines=8, show_copy_button=True)
            img_btn.click(fn=ocr_easyocr_tab, inputs=[img_input, img_langs],
                          outputs=[img_output, img_detections, img_details])

        # ─── Tab 3: Handwriting OCR ────────────────────────────────
        with gr.Tab("✍️ Handwriting OCR"):
            gr.Markdown("### Upload handwritten text for recognition")
            with gr.Row():
                with gr.Column():
                    hw_input = gr.Image(label="Upload Handwritten Image", type="pil")
                    hw_btn = gr.Button("Recognize", variant="primary")
                with gr.Column():
                    hw_details = gr.Markdown(label="Details")
            hw_output = gr.Textbox(label="Recognized Text", lines=8, show_copy_button=True)
            hw_btn.click(fn=ocr_trocr_tab, inputs=[hw_input], outputs=[hw_output, hw_details])

        # ─── Tab 4: Spell Correction ────────────────────────────────
        with gr.Tab("✏️ Spell Correction"):
            gr.Markdown("### Correct spelling errors")
            with gr.Row():
                with gr.Column():
                    spell_input = gr.Textbox(label="Input Text", lines=6,
                                            placeholder="Enter text with spelling errors...")
                    spell_lang = gr.Radio(choices=["Arabic", "English"],
                                          value="English", label="Language")
                    spell_btn = gr.Button("Correct", variant="primary")
                with gr.Column():
                    spell_details = gr.Markdown(label="Details")
            spell_output = gr.Textbox(label="Corrected Text", lines=6, show_copy_button=True)
            spell_btn.click(fn=correct_spelling_tab,
                            inputs=[spell_input, spell_lang],
                            outputs=[spell_output, spell_details])

        # ─── Tab 5: Translation ────────────────────────────────────
        with gr.Tab("🌐 Translation"):
            gr.Markdown("### Translate between English and Arabic")
            with gr.Row():
                with gr.Column():
                    trans_input = gr.Textbox(label="Input Text", lines=6,
                                            placeholder="Enter text to translate...")
                    trans_dir = gr.Radio(
                        choices=["English → Arabic", "Arabic → English"],
                        value="English → Arabic", label="Direction"
                    )
                    trans_btn = gr.Button("Translate", variant="primary")
                with gr.Column():
                    trans_details = gr.Markdown(label="Details")
            trans_output = gr.Textbox(label="Translation", lines=6, show_copy_button=True)
            trans_btn.click(fn=translate_tab,
                            inputs=[trans_input, trans_dir],
                            outputs=[trans_output, trans_details])

        # ─── Tab 6: Evaluation ─────────────────────────────────────
        with gr.Tab("📊 Evaluation"):
            gr.Markdown("### Calculate CER & WER metrics")
            with gr.Row():
                with gr.Column():
                    eval_ref = gr.Textbox(label="Reference Text (ground truth)",
                                          lines=4, placeholder="Enter reference text...")
                    eval_hyp = gr.Textbox(label="Hypothesis Text (OCR output)",
                                          lines=4, placeholder="Enter hypothesis text...")
                    eval_btn = gr.Button("Evaluate", variant="primary")
                with gr.Column():
                    eval_output = gr.Markdown(label="Metrics")
            eval_diff = gr.Code(label="Diff", language="diff")
            eval_btn.click(fn=evaluate_tab,
                            inputs=[eval_ref, eval_hyp],
                            outputs=[eval_output, eval_diff])

        # ─── Tab 7: About ──────────────────────────────────────────
        with gr.Tab("ℹ️ About"):
            gr.Markdown(about_content)


# ═══════════════════════════════════════════════════════════════════
# Launch with ngrok
# ═══════════════════════════════════════════════════════════════════

print("\n🌐 Setting up ngrok tunnel...")

try:
    from pyngrok import ngrok

    # Kill any existing tunnels
    ngrok.kill()

    # Set auth token (optional — use Colab secrets for security)
    NGROK_AUTH_TOKEN = os.environ.get("NGROK_AUTH_TOKEN", "")
    if NGROK_AUTH_TOKEN:
        ngrok.set_auth_token(NGROK_AUTH_TOKEN)
        print("   ✅ ngrok auth token configured")
    else:
        print("   ℹ️  No ngrok auth token — using free tier")
        print("      (Set NGROK_AUTH_TOKEN in Colab secrets for better stability)")

    # Create tunnel
    public_url = ngrok.connect(7860)
    print(f"\n{'=' * 70}")
    print(f"  🌐 Gradio App URL: {public_url}")
    print(f"{'=' * 70}")
    print(f"\n   👆 Click the URL above to open the app in a new tab!")
    print(f"   The app will be available as long as this cell is running.\n")

except Exception as e:
    print(f"   ⚠️  ngrok setup: {e}")
    print(f"   The app will use the default Colab preview URL.")

# Launch Gradio
print("\n🚀 Launching Gradio on port 7860...")
demo.launch(
    server_name="0.0.0.0",
    server_port=7860,
    share=False,  # Using ngrok instead
    show_error=True,
)


---

## ✅ Congratulations!

You have completed the **OmniFile AI Processor v4.0** cloud workbook.

### 📋 What You've Accomplished

| Step | Feature | Status |
|------|---------|--------|
| 1 | Environment Setup | ✅ |
| 2 | Google Drive Integration | ✅ |
| 3 | OCR Engine Initialization | ✅ |
| 4 | PDF Processing | ✅ |
| 5 | Image OCR | ✅ |
| 6 | Spell Correction | ✅ |
| 7 | Translation | ✅ |
| 8 | Evaluation Metrics | ✅ |
| 9 | Gradio Web UI | ✅ |

---

### 🚀 Next Steps

1. **Save Results** — All processed data is saved in `/content/drive/MyDrive/OmniFile_AI/`
2. **Customize Models** — Fine-tune TrOCR or EasyOCR for your specific use case
3. **Add Languages** — Extend OCR support by adding more language models
4. **Deploy** — Deploy the Gradio app on HuggingFace Spaces for permanent access
5. **Contribute** — Star the repo and submit pull requests!

---

### 🔗 Important Links

| Resource | URL |
|---|---|
| **GitHub Repository** | [https://github.com/DrAbdulmalek/OmniFile_Processor](https://github.com/DrAbdulmalek/OmniFile_Processor) |
| **HuggingFace Spaces** | [https://huggingface.co/spaces/DrAbdulmalek/handwriting-ocr](https://huggingface.co/spaces/DrAbdulmalek/handwriting-ocr) |

---

> **💡 Tip:** To keep your models cached between sessions, make sure the `models_cache` folder
> in your Google Drive is preserved. This saves significant download time on restart.

---

### 📝 License

This project is developed by **Dr. Abdulmalek**. Please refer to the GitHub repository for license details.
